# Extrator do Lacuna com QLoRA

Este notebook ajusta o Qwen2.5-1.5B-Instruct para ler um extrato de contrato de diário oficial e devolver o JSON do esquema `Ato` do Lacuna: tipo, número, processo, contratada, CNPJ, objeto, valor e data de assinatura. O objetivo é ver o quanto um modelo de 1,5B, ajustado com 289 exemplos numa T4 gratuita, chega perto dos LLMs grandes no mesmo conjunto ouro.

O treino usa `data/treino/rotulos_claude.jsonl`: 289 trechos de 13 municípios do RJ, rotulados por LLM e sem revisão humana, sem nenhum trecho que tenha o mesmo texto ou o mesmo contrato do conjunto ouro. A avaliação usa `data/gold/dev.jsonl` (40) e `data/gold/test.jsonl` (60), revisados à mão, com a mesma função `score` do regex e dos LLMs via API.

Para rodar, abra no Colab e escolha a GPU T4 em Ambiente de execução → Alterar o tipo de ambiente de execução.

In [ ]:
!rm -rf /content/lacuna && git clone -q https://github.com/juan-barross/lacuna.git /content/lacuna
%cd /content/lacuna
!pip install -q -e . "accelerate==1.15.0" "datasets==5.1.0"
!pip install -q "transformers==5.22.1" "peft==0.21.2" "trl==1.17.1" "bitsandbytes==0.50.2"

## Configuração

Os valores do experimento ficam aqui. Com `RODADA_FINAL = True`, o notebook mede o dev e o test, para o model card. Para comparar configurações sem olhar o test, use `False`.

O login pede um token do Hugging Face com permissão de escrita (Settings → Access Tokens). Ele vem logo no início para o adaptador subir assim que o treino acabar.

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
RUN = "qwen2.5-1.5b-r16-e3"
SEED = 2027
MAX_LENGTH = 2048
RODADA_FINAL = True
USUARIO_HF = "SEU_USUARIO_HF"
REPO = f"{USUARIO_HF}/lacuna-extrator-qwen2.5-1.5b-lora"

LORA = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
TREINO = dict(
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
)

assert USUARIO_HF != "SEU_USUARIO_HF", "troque USUARIO_HF pelo seu usuário do Hugging Face"

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Dados

O prompt é o `LOCAL_PROMPT` do Lacuna: o mesmo `PROMPT` dos LLMs via API mais o esquema JSON do `Ato`, que pela API vai no `response_format`. Assim o Qwen recebe a mesma informação que o Opus e o Nemotron. O alvo de cada exemplo é o JSON do rótulo.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "src")

from datasets import Dataset

from dxp.evaluation import score
from dxp.files import read_jsonl
from dxp.llm import PROMPT_VERSION, local_messages
from dxp.schema import Ato


def alvo(row):
    return json.dumps(Ato.model_validate(row).model_dump(mode="json"), ensure_ascii=False)


treino = read_jsonl(Path("data/treino/rotulos_claude.jsonl"))
ouro = {split: read_jsonl(Path(f"data/gold/{split}.jsonl")) for split in ("dev", "test")}
assert not {r["text"] for r in treino} & {r["text"] for g in ouro.values() for r in g}, (
    "trecho do ouro no treino"
)

exemplos = [
    {"prompt": local_messages(r["text"]), "completion": [{"role": "assistant", "content": alvo(r)}]}
    for r in treino
]
ds = Dataset.from_list(exemplos)
print(ds)
print(exemplos[0]["completion"][0]["content"])

## Modelo em 4 bits

O QLoRA carrega o modelo base quantizado em NF4 e treina só os adaptadores LoRA. A T4 não tem bf16, por isso o cálculo é em fp16. O `assert` garante que nenhum exemplo é cortado pelo `MAX_LENGTH`, o que tiraria o fim do JSON do treino.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE)
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=quant, device_map={"": 0}, dtype=torch.float16
)

tamanhos = sorted(
    len(tok.apply_chat_template(e["prompt"] + e["completion"], return_dict=True)["input_ids"])
    for e in exemplos
)
print(f"tokens por exemplo: mediana {tamanhos[len(tamanhos) // 2]}, máximo {tamanhos[-1]}")
assert tamanhos[-1] <= MAX_LENGTH

## Como o extrator é medido

A resposta do modelo passa pelo `LocalExtractor` do Lacuna (`dxp.local_model`), que aplica o `parse_answer`. Quando não é um JSON válido do `Ato`, o trecho vira `Ato(tipo="outro")`, a mesma regra do extrator via API, e conta como erro. A geração é gulosa (sem amostragem), então o resultado se repete. A latência é da T4 com a base em 4 bits e não se compara com a das APIs.

In [ ]:
import time

from dxp.local_model import LocalExtractor


def avaliar(model, split, nome):
    model.eval()
    extrator = LocalExtractor(tok, model)
    campos = score(ouro[split], extrator)
    return {"campos": campos, "uso": {**extrator.usage(nome), "gpu": torch.cuda.get_device_name()}}


def resumo(resultado):
    return {campo: round(m["f1"], 2) for campo, m in resultado["campos"].items()}


SPLITS = ("dev", "test") if RODADA_FINAL else ("dev",)

## O modelo sem ajuste

O mesmo Qwen, sem LoRA e com o mesmo prompt, mostra o quanto o ajuste acrescenta.

In [ ]:
resultados = {}
for split in SPLITS:
    resultados[f"base_{split}"] = avaliar(model, split, BASE)
    print(split, resumo(resultados[f"base_{split}"]))

## Treino

Os 289 exemplos entram todos no treino, porque o dev do conjunto ouro já faz o papel de validação. A perda é calculada só na resposta (`completion`). O adaptador sobe para o Hugging Face, como privado, logo depois do treino, para não se perder se o Colab desconectar durante a avaliação.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=f"runs/{RUN}",
    seed=SEED,
    max_length=MAX_LENGTH,
    fp16=True,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    **TREINO,
)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds,
    processing_class=tok,
    peft_config=LoraConfig(task_type="CAUSAL_LM", **LORA),
)
trainer.model.print_trainable_parameters()
inicio = time.perf_counter()
trainer.train()
minutos_treino = round((time.perf_counter() - inicio) / 60, 1)
trainer.model.save_pretrained(f"adapters/{RUN}")
tok.save_pretrained(f"adapters/{RUN}")
trainer.model.push_to_hub(REPO, private=True)
tok.push_to_hub(REPO, private=True)
perda_final = [r["loss"] for r in trainer.state.log_history if "loss" in r][-1]
print(f"treino: {minutos_treino} min, perda final {perda_final:.3f}")

## Avaliação no conjunto ouro

In [ ]:
from datetime import date

Path("reports").mkdir(exist_ok=True)
for split in SPLITS:
    resultados[split] = avaliar(trainer.model, split, f"{BASE} + LoRA ({RUN})")
    print(split, resumo(resultados[split]), resultados[split]["uso"])
    Path(f"reports/{date.today()}_qlora_{split}.json").write_text(
        json.dumps(resultados[split], indent=2, ensure_ascii=False), encoding="utf-8"
    )

## Comparação com os outros extratores

Lê os relatórios em `reports/` e põe o F1 de cada campo lado a lado. O Opus só tem relatório no dev, porque processou 45 dos 60 trechos do test.

In [ ]:
import pandas as pd


def tabela(split):
    linhas = {}
    for path in sorted(Path("reports").glob(f"*_{split}.json")):
        relatorio = json.loads(path.read_text(encoding="utf-8"))
        if "campos" in relatorio:
            linhas[path.stem.split("_", 1)[1]] = resumo(relatorio)
    linhas[f"qwen sem ajuste_{split}"] = resumo(resultados[f"base_{split}"])
    return pd.DataFrame(linhas).T


for split in SPLITS:
    display(tabela(split))

## Histórico de execuções

Uma linha por execução, com o commit, a versão do prompt, a configuração e o F1, para comparar r=8 com r=16 ou 0,5B com 1,5B.

In [ ]:
import subprocess

import bitsandbytes
import peft
import transformers
import trl

linha = {
    "run": RUN,
    "data": str(date.today()),
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "prompt": PROMPT_VERSION,
    "base": BASE,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "lora": LORA,
    "treino": TREINO,
    "exemplos_treino": len(ds),
    "minutos_treino": minutos_treino,
    "perda_final": perda_final,
    "f1": {split: resumo(resultados[split]) for split in SPLITS},
    "erros_json": {split: resultados[split]["uso"]["erros"] for split in SPLITS},
    "versoes": {lib.__name__: lib.__version__ for lib in (torch, transformers, peft, trl, bitsandbytes)},
}
Path("runs").mkdir(exist_ok=True)
with open("runs/historico.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(linha, ensure_ascii=False) + "\n")
linha

## Salvar

Os relatórios e o histórico vão para download, para entrarem no repositório. Confira o model card antes de tornar o adaptador público.

In [ ]:
from google.colab import files

!zip -qr resultados_{RUN}.zip reports runs/historico.jsonl
files.download(f"resultados_{RUN}.zip")